<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/TDConnect4Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitbully[gui]

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

# Export Legacy Format to Standardized format:

In [ ]:
from techdays26.legacy_ntuple_agent import (
    TDWeightsLoader,
    export_two_player_weights_zip,
    import_two_player_weights_zip,
)
from techdays26.legacy_ntuple_agent import TDConnect4Agent, TDEvaluator
import numpy as np

In [ ]:
# =============================================================================
# Example usage for legacy weights in ZIP file (including Protocol check)
# =============================================================================
if False:
    loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)

    # Directory:
    # both = loader.load_two_player(".")

    # Zip:
    both = loader.load_two_player_from_zip(
        "C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt"
    )

    evaluator = TDEvaluator(both)
    td_agent = TDConnect4Agent(evaluator, tie_break="center")

In [ ]:
if True:
    loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)
    both = loader.load_two_player_from_zip(
        "C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt"
    )

    # Export to a clean single file:
    export_two_player_weights_zip("td_weights_clean.tdw.zip", both)

evaluator = TDEvaluator(
    import_two_player_weights_zip("td_weights_clean.tdw.zip", int_dtype=np.int32)
)
td_agent = TDConnect4Agent(evaluator, tie_break="center")

In [ ]:
# Structural/runtime Protocol check (optional):
from bitbully.agent_interface import Connect4Agent

assert isinstance(td_agent, Connect4Agent)

In [ ]:
from bitbully import BitBully

bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
%matplotlib ipympl
from bitbully.gui_c4 import GuiC4

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
from techdays26.legacy import play_match

result = play_match(td_agent, bitbully_agent, verbose=1)
# 0 draw, 1 yellow win, 2 red win
print(result)

In [ ]:
from collections import defaultdict

n_matches = 200
total_scores = defaultdict(int)
for _ in range(n_matches):
    result = play_match(td_agent, bitbully_agent, verbose=0)
    total_scores[result] += 1

In [ ]:
total_scores

In [ ]:
import logging
from techdays26 import bitbully_arena as ba

logger = logging.getLogger("bitbully.arena")

# ── Define your agents ──────────────────────────────────────────────
random_spec = ba.AgentSpec(
    agent_id="random",
    agent=ba.RandomAgent(),
    epsilons=(0.0,),
)

bitbully_spec = ba.AgentSpec(
    agent_id="bitbully",
    agent=bitbully_agent,  # your BitBully agent instance
    epsilons=(0.0, 0.10),
)

td_spec = ba.AgentSpec(
    agent_id="td",
    agent=td_agent,  # your TD agent instance
    epsilons=(0.0, 0.05),
)

# ── Option A: round-robin (original behavior, matchups=None) ───────
cfg_roundrobin = ba.ArenaConfig(
    agents=(random_spec, bitbully_spec, td_spec),
    n_games=50,
    # matchups omitted → all-vs-all, both color assignments
    seed=42,
    use_tqdm=True,
    logger=logger,
)

# ── Option B: explicit constellations ──────────────────────────────
cfg_explicit = ba.ArenaConfig(
    agents=(random_spec, bitbully_spec, td_spec),
    n_games=5,
    matchups=(
        ba.Matchup(yellow_id="bitbully", red_id="random"),  # bitbully starts
        ba.Matchup(yellow_id="td", red_id="bitbully"),  # td starts vs bitbully
        ba.Matchup(yellow_id="bitbully", red_id="td"),  # bitbully starts vs td
    ),
    seed=42,
    use_tqdm=True,
    logger=logger,
)

# ── Run & inspect ──────────────────────────────────────────────────
arena = ba.BitBullyArena()
result = arena.run(cfg_explicit)

print(ba.format_aggregate_table(result))
print(ba.get_table_legend())

# Optionally persist:
# result.save_json("arena_result.json")